# RNA-seq: differential expression from a count table

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fbcouto/multimerge-genomics-research/blob/main/notebooks/rna_seq/differential_expression.ipynb)

**The problem.** You have read counts per gene for each sample (from featureCounts, htseq-count, nf-core
rnaseq or a public repository) and a table that says which sample got which treatment. Which genes change
with the treatment, beyond the noise between replicates, and while accounting for a pairing or batch
factor (the same donor, cell line or run in both groups)? This notebook applies the DESeq2 method.

**What you need.** A count table (a gene column, then one column per sample: raw integer counts, not
normalized values); a sample table (a sample column, then one column per factor); the factor to test
and its reference level; optionally a table of gene names.

**What you get.** A results table for every gene (base mean, log2 fold change, standard error, p-value,
adjusted p-value), the number of genes up and down, the top genes, and volcano and MA plots.

In [ ]:
# Setup. Runs on Google Colab or on your own computer: on Colab, the first run installs MMGEN from PyPI
# (about a minute). Tables use pandas and plots matplotlib when installed (Colab has both); without
# them the notebook prints plain text and skips the plots.
import platform, subprocess, sys
from pathlib import Path

RELEASE = "https://github.com/fbcouto/multimerge-genomics-research/releases/download/v2.5.1"      # the demo packs
PACKAGE = "multimerge-genomics==2.5.1"   # from PyPI
try:
    import multimerge_genomics as mmg
except ImportError:
    if platform.system() != "Linux":
        print("MMGEN is not installed in this Python. Install it with:\n"
              "  python -m pip install multimerge-genomics\n"
              "and run this cell again.")
        raise
    print("Installing MMGEN (about a minute)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", PACKAGE], check=True)
    import multimerge_genomics as mmg

IN_NOTEBOOK = "ipykernel" in sys.modules
try:
    from IPython.display import display, Markdown
except ImportError:                        # plain Python: print instead
    display, Markdown = print, str
try:
    import pandas as pd
    pd.set_option("display.width", 160)
    pd.set_option("display.max_columns", 30)
except ImportError:
    pd = None
try:
    import matplotlib
    if not IN_NOTEBOOK:
        matplotlib.use("Agg")              # plain Python: save the figures, open no window
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

# Chart colors (a palette checked for color-blind readers): two series, a two-sided scale, neutral ink
BLUE, ORANGE, RED, GRAY, INK = "#2a78d6", "#eb6834", "#e34948", "#a3a29c", "#52514e"


def show(rows, n=25):
    """Shows a list of dictionaries as a table (pandas when available, plain text otherwise)."""
    rows = list(rows)
    if pd is not None:
        display(pd.DataFrame(rows[:n]))
    else:
        for row in rows[:n]:
            print("  ".join(f"{k}: {v}" for k, v in row.items()))
    if len(rows) > n:
        print(f"... and {len(rows) - n} more rows")


def tidy(ax):
    """Recessive axes: no top or right frame, a light grid behind the data."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.set_axisbelow(True)


def save_figure(fig, path):
    """Saves a figure; shows it in a notebook, closes it in plain Python."""
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("Figure saved to", path)
    if IN_NOTEBOOK:
        plt.show()
    else:
        plt.close(fig)


print(mmg.engine_info())

## Data

By default, from the research demo pack (downloaded once, about 45 MB): the airway experiment (Himes et al. 2014, GEO
GSE52778), airway smooth-muscle cells of four donors (cell lines), each untreated and treated with the
corticosteroid dexamethasone; raw counts per gene computed by NCBI on GRCh38, and the NCBI gene names.
The count table also holds samples treated with albuterol: the sample table lists only the eight to
compare, and the other columns are ignored. To use your own data, set the `MY_...` paths and the design.

In [ ]:
# Your tables (None = the demo data) and the design
MY_COUNTS = None             # TSV: gene, then one column of raw counts per sample (.gz allowed)
MY_SAMPLES = None            # TSV: sample, then one column per factor
MY_GENE_NAMES = None         # optional: NCBI gene_info, or a two-column table (id, name)
CONDITION = "dex"            # the factor to test (a column of the sample table)
REFERENCE_LEVEL = "untrt"    # its baseline level; the other level is the "treated" one
BLOCK_BY = ["cell_line"]     # factors to account for (pairing, batch); [] for none
ALPHA = 0.1                  # false discovery rate for calling a gene changed

import urllib.request, zipfile

DATA = Path("data")                          # the demo files are extracted here
OUT = Path("results") / "differential_expression"         # everything this notebook writes
OUT.mkdir(parents=True, exist_ok=True)


def demo_file(pack, member):
    """One file of a demo pack: downloads the pack once into the working folder and extracts only that file."""
    target = DATA / member
    if not target.exists():
        if not Path(pack).exists():
            print(f"Downloading {pack} from the MMGEN release...")
            urllib.request.urlretrieve(f"{RELEASE}/{pack}", pack + ".part")
            Path(pack + ".part").replace(pack)
        with zipfile.ZipFile(pack) as z:
            z.extract(member, DATA)
    return str(target)


USING_DEMO = MY_COUNTS is None
COUNTS = MY_COUNTS or demo_file("mmgen_demo_research_v4.zip", "research/rnaseq/airway_counts.tsv.gz")
SAMPLES = MY_SAMPLES or demo_file("mmgen_demo_research_v4.zip", "research/rnaseq/airway_samples.tsv")
GENE_NAMES = demo_file("mmgen_demo_research_v4.zip", "research/rnaseq/airway_gene_names.tsv") if USING_DEMO else MY_GENE_NAMES
print("Counts:", COUNTS)
print("Samples:", SAMPLES)

## Step 1. Look at the inputs

Check that every sample of the design is a column of the count table, that the groups are balanced, and
that the library sizes (total counts) are of the same order: DESeq2 corrects for depth, but a sample with
10 times fewer reads than the others, or with almost none, deserves a look before the test.

In [ ]:
import csv, gzip

with open(SAMPLES, newline="") as f:
    design = list(csv.DictReader(f, delimiter="\t"))
sample_column = list(design[0])[0]
names = [row[sample_column] for row in design]

opener = gzip.open if COUNTS.endswith(".gz") else open
with opener(COUNTS, "rt") as f:
    lines = (line for line in f if not line.startswith("#"))       # featureCounts writes a comment first
    header = next(lines).rstrip("\n").split("\t")
    columns = {name: header.index(name) for name in names if name in header}
    totals, n_genes = dict.fromkeys(columns, 0.0), 0
    for line in lines:
        values = line.rstrip("\n").split("\t")
        for name, i in columns.items():
            totals[name] += float(values[i])
        n_genes += 1

print(f"{n_genes:,} genes; {len(header) - 1} columns in the count table, {len(names)} samples in the design")
show({**row, "in count table": row[sample_column] in columns,
      "million counts": round(totals.get(row[sample_column], 0) / 1e6, 2)} for row in design)
missing = [n for n in names if n not in columns]
if missing:
    raise RuntimeError(f"Samples of the design missing from the count table: {missing}")

## Step 2. Test every gene

The DESeq2 method: each sample's depth is estimated by the median of ratios to a pseudo-reference (size
factors); each gene's dispersion (its biological variability) is estimated and shrunk toward a trend over
the mean, which matters with few replicates; a negative binomial model with the blocking factors and the
condition is fitted; and a Wald test gives each gene's p-value. Adjusted p-values (Benjamini-Hochberg)
control the false discovery rate; genes with too few counts to ever be significant are left out of the
adjustment (independent filtering), and outliers are flagged by Cook's distance.

In [ ]:
de = mmg.differential_expression(COUNTS, SAMPLES, CONDITION, REFERENCE_LEVEL, output=str(OUT / "de_results.tsv"),
                                 factors=BLOCK_BY or None, alpha=ALPHA, gene_names=GENE_NAMES)
print(de["text"])
for note in de["warnings"]:
    if note not in de["text"]:
        print("Note:", note)

## Step 3. The top genes, volcano and MA plots

Sorted by adjusted p-value. A large fold change on a low mean count is less reliable than a moderate one
on a highly expressed gene: look at `base mean` too.

The volcano plot shows effect (log2 fold change) against evidence (-log10 adjusted p-value); the MA plot
shows effect against mean expression, where low-count genes spread widely. Genes called changed are
colored (up red, down blue); the ten most significant are named.

In [ ]:
p = de["plot"]
genes = [{"gene": p["gene"][i], "name": (p.get("name") or p["gene"])[i], "base mean": round(p["base_mean"][i], 1),
          "log2 fold change": None if p["log2_fold_change"][i] is None else round(p["log2_fold_change"][i], 3),
          "padj": p["padj"][i]} for i in range(len(p["gene"]))]
significant = sorted((g for g in genes if g["padj"] is not None and g["padj"] < ALPHA), key=lambda g: g["padj"])
print(f"{len(significant):,} genes with padj < {ALPHA}: {de['up']:,} up and {de['down']:,} down in {de['treated']} against {de['reference']}")
show(significant[:15])

if USING_DEMO:     # classic glucocorticoid-response genes (Himes et al. 2014)
    by_name = {g["name"]: g for g in genes}
    show(by_name[n] for n in ("FKBP5", "TSC22D3", "DUSP1", "KLF15", "ZBTB16", "PER1") if n in by_name)

import math

if plt is not None:
    kind = lambda g: "not changed" if g["padj"] is None or g["padj"] >= ALPHA else ("up" if g["log2 fold change"] > 0 else "down")
    colors = {"not changed": GRAY, "up": RED, "down": BLUE}
    plotted = [g for g in genes if g["padj"] is not None and g["log2 fold change"] is not None]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))
    for k in ("not changed", "down", "up"):
        pts = [g for g in plotted if kind(g) == k]
        ax1.scatter([g["log2 fold change"] for g in pts], [-math.log10(max(g["padj"], 1e-300)) for g in pts],
                    s=5, color=colors[k], linewidths=0, label=f"{k} ({len(pts):,})")
        ax2.scatter([math.log10(g["base mean"]) for g in pts], [g["log2 fold change"] for g in pts],
                    s=5, color=colors[k], linewidths=0, label=f"{k} ({len(pts):,})")
    # name the ten most significant genes; of names at about the same height, the leftmost point's goes left
    top = sorted(((g["log2 fold change"], -math.log10(max(g["padj"], 1e-300)), g["name"]) for g in significant[:10]),
                 key=lambda t: -t[1])
    for i, (x, y, name) in enumerate(top):
        row = [t for t in top if abs(t[1] - y) < 0.02 * top[0][1]]
        left = len(row) > 1 and x == min(t[0] for t in row)
        ax1.annotate(name, (x, y), xytext=(-4 if left else 4, 0), textcoords="offset points",
                     ha="right" if left else "left", va="center", fontsize=7, color=INK)
    ax1.set(title="Volcano plot", xlabel="log2 fold change", ylabel="-log10 adjusted p-value")
    ax2.axhline(0, color=INK, linewidth=0.8)
    ax2.set(title="MA plot", xlabel="log10 mean of normalized counts", ylabel="log2 fold change")
    for ax in (ax1, ax2):
        ax.legend(frameon=False, fontsize=8, markerscale=3)
        tidy(ax)
    fig.tight_layout()
    save_figure(fig, OUT / "volcano_ma.png")

## Check the result

In [ ]:
def check(condition, text):
    print(("OK     " if condition else "CHECK  ") + text)


factors = dict(zip(de["samples"], de["size_factors"]))
check(all(0.25 < f < 4 for f in factors.values()), "size factors are within 0.25-4 (no sample with a very different depth)")
check(de["not_converged"] == 0, f"the model converged for every gene ({de['not_converged']} did not)")
check(de["outliers"] < 0.01 * de["tested"], f"few outlier genes by Cook's distance ({de['outliers']})")
check(de["up"] + de["down"] > 0, "some genes are called changed")
check(Path(de["output"]).exists(), f"the results table is at {de['output']}")
if USING_DEMO:
    assert de["up"] > 1000 and de["down"] > 1000
    assert all(by_name[n]["padj"] < ALPHA and by_name[n]["log2 fold change"] > 1 for n in ("FKBP5", "TSC22D3", "DUSP1", "KLF15", "ZBTB16"))
    print("Demo expectations met.")

## Methods paragraph

A starting text for the methods section of a paper or report, filled in with the version and the parameters
this run used. Edit it to your study, and cite the methods MMGEN reproduces (see References).

In [ ]:
import textwrap
VERSION = mmg.engine_info()["version"]
methods = f"""Differential expression between {de['treated']} and {de['reference']} ({de['condition']}) was tested
with the DESeq2 method as implemented in MMGEN version {VERSION} (Couto F. MMGEN: MultiMerge Genomics (version {VERSION}). Zenodo. https://doi.org/10.5281/zenodo.20417492), with the design
{de['design']}: size factors by the median of ratios, gene-wise dispersions shrunk toward a parametric trend, a
negative binomial generalized linear model and a Wald test. P-values were adjusted by the Benjamini-Hochberg method
with independent filtering by mean count, and outliers were flagged by Cook's distance. Of {de['tested']:,} genes
with non-zero counts, {de['up']:,} were up and {de['down']:,} down at an adjusted p-value below {ALPHA}."""
print(textwrap.fill(" ".join(methods.split()), 110))

## Adapt it to your data

* Use raw counts. TPM, FPKM or normalized values break the model.
* No pairing or batch: `BLOCK_BY = []`. More than two levels in the condition: give `treated=` to
  `differential_expression` to pick the level compared with the reference.
* Need a minimum effect, not only a non-zero one: `lfc_threshold=1` tests |log2 fold change| > 1.
* From BAM files: `mmg.count_reads(bams, "annotation.gtf", "counts.tsv")` makes the count table first.
* Results can differ slightly from DESeq2 in R; for a paper, state which implementation you used.

## References

* Love M.I., Huber W., Anders S. (2014) Moderated estimation of fold change and dispersion for RNA-seq data
  with DESeq2. *Genome Biology* 15:550.
* Anders S., Huber W. (2010) Differential expression analysis for sequence count data. *Genome Biology* 11:R106.
* Benjamini Y., Hochberg Y. (1995) Controlling the false discovery rate. *J. R. Stat. Soc. B* 57:289.
* Bourgon R., Gentleman R., Huber W. (2010) Independent filtering increases detection power for high-throughput
  experiments. *PNAS* 107:9546.
* Himes B.E. et al. (2014) RNA-Seq transcriptome profiling identifies CRISPLD2 as a glucocorticoid responsive
  gene that modulates cytokine function in airway smooth muscle cells. *PLoS One* 9:e99625.